# Odin: Orca image normalization workflow

This notebook shows how to use the workflow to compute normalized images recorded by the Orca detector on the ODIN instrument.

In [ ]:
import ess.odin.data  # noqa: F401
from ess import odin
from ess.imaging.types import *
import scipp as sc
import plopp as pp

%matplotlib widget

## Workflow setup

In [ ]:
wf = odin.OdinOrcaWorkflow()

wf[Filename[AllRuns]] = odin.data.odin_lego_images()

wf[NeXusDetectorName] = 'histogram_mode_detectors/orca'

wf[MaskingRules] = {}  # No masks to begin with
wf[UncertaintyBroadcastMode] = UncertaintyBroadcastMode.upper_bound

In [ ]:
wf.visualize(NormalizedImage, compact=True, graph_attr={"rankdir": "LR"})

## Run the workflow

We compute the final normalized image:

In [ ]:
image = wf.compute(NormalizedImage)
image

In [ ]:
pp.slicer(image, autoscale=False)

## Adding masks

If we want to mask some part of the image, we update the masking rules.
For example, here we mask the upper part of the image:

In [ ]:
wf[MaskingRules] = {'y_pixel_offset': lambda x: x > sc.scalar(0.082, unit='m')}

pp.slicer(wf.compute(NormalizedImage), autoscale=False)

## Intermediate results

We can also inspect intermediate results, which is useful for debugging:

In [ ]:
results = wf.compute(
    [
        FluxNormalizedDetector[SampleRun],
        FluxNormalizedDetector[OpenBeamRun],
        BackgroundSubtractedDetector[SampleRun],
    ]
)

fig = pp.tiled(2, 2, hspace=0.3, wspace=0.3)
fig[0, 0] = results[FluxNormalizedDetector[SampleRun]]['time', 0].plot(
    title='Sample (proton-charge normalized)'
)
fig[0, 1] = results[FluxNormalizedDetector[OpenBeamRun]]['time', 0].plot(
    title='Open beam (proton-charge normalized)'
)
fig[1, 0] = results[BackgroundSubtractedDetector[SampleRun]]['time', 0].plot(
    title='Background subtracted sample'
)
fig[1, 1] = image['time', 0].plot(title='Final image')
fig